# 01 — Creación del Dataset Maestro Granada

Objetivo: construir una primera versión del dataset maestro del proyecto **CoopScore Granada** a partir de datos GIS de suelo residencial y datos oficiales de población municipal.

Entradas esperadas:

- `data/raw/suelo_granada.csv`
- `data/raw/suelo_granada_clasificado.csv`
- `data/raw/suelo_granada_residencial.csv`
- `data/raw/2871.xlsx`

Salidas generadas:

- `data/processed/municipios_granada.csv`
- `data/processed/suelo_residencial_municipio.csv`
- `data/processed/dataset_maestro_granada_v1.csv`


In [ ]:
# ============================================================
# 0. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Rutas del proyecto
BASE_DIR = Path.cwd()
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Directorio base:', BASE_DIR)
print('Raw existe:', DATA_RAW.exists())
print('Processed existe:', DATA_PROCESSED.exists())

In [ ]:
# ============================================================
# 1. CARGA DE DATOS RAW
# ============================================================

suelo = pd.read_csv(DATA_RAW / 'suelo_granada.csv')
suelo_clasificado = pd.read_csv(DATA_RAW / 'suelo_granada_clasificado.csv')
suelo_residencial = pd.read_csv(DATA_RAW / 'suelo_granada_residencial.csv')
ine_raw = pd.read_excel(DATA_RAW / '2871.xlsx', header=None)

print('suelo:', suelo.shape)
print('suelo_clasificado:', suelo_clasificado.shape)
print('suelo_residencial:', suelo_residencial.shape)
print('ine_raw:', ine_raw.shape)

In [ ]:
# Vista rápida de los datos GIS residenciales
suelo_residencial.head()

In [ ]:
# Vista rápida del Excel INE
ine_raw.head(15)

In [ ]:
# ============================================================
# 2. LIMPIEZA DEL EXCEL INE: CÓDIGO INE, MUNICIPIO Y POBLACIÓN
# ============================================================

# En el Excel, a partir de la fila 8 aparecen los municipios.
# Columna 0: '18001 Agrón'
# Columna 1: población total 2025

ine = ine_raw.iloc[8:, [0, 1]].copy()
ine.columns = ['municipio_raw', 'poblacion_2025']

# Eliminamos filas vacías o no municipales
ine = ine.dropna(subset=['municipio_raw'])
ine['municipio_raw'] = ine['municipio_raw'].astype(str).str.strip()

# Nos quedamos solo con filas que empiezan por 5 dígitos INE
ine = ine[ine['municipio_raw'].str.match(r'^\d{5}\s+')].copy()

# Extraemos código y nombre
ine['codigo_ine'] = ine['municipio_raw'].str.extract(r'^(\d{5})')
ine['municipio'] = ine['municipio_raw'].str.replace(r'^\d{5}\s+', '', regex=True).str.strip()

# Convertimos población a numérico
ine['poblacion_2025'] = (
    ine['poblacion_2025']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
ine['poblacion_2025'] = pd.to_numeric(ine['poblacion_2025'], errors='coerce')

municipios = ine[['codigo_ine', 'municipio', 'poblacion_2025']].copy()
municipios.head()

In [ ]:
# Comprobación básica de municipios
print('Municipios INE cargados:', municipios.shape[0])
print('Duplicados codigo_ine:', municipios['codigo_ine'].duplicated().sum())
municipios.sample(10, random_state=42)

In [ ]:
# ============================================================
# 3. LIMPIEZA Y RESUMEN DEL SUELO RESIDENCIAL
# ============================================================

res = suelo_residencial.copy()

# Normalizamos código INE a string de 5 dígitos
res['codigo_ine'] = res['codigo_ine'].astype(int).astype(str).str.zfill(5)

# Renombramos columnas técnicas
res = res.rename(columns={
    'SHAPE.STLength()': 'perimetro_m',
    'area_m2': 'superficie_poligono_m2'
})

# Aseguramos tipos numéricos
res['superficie_poligono_m2'] = pd.to_numeric(res['superficie_poligono_m2'], errors='coerce')
res['perimetro_m'] = pd.to_numeric(res['perimetro_m'], errors='coerce')

# Validaciones rápidas
print('Registros residenciales:', len(res))
print('Municipios con suelo residencial:', res['codigo_ine'].nunique())
print('Área total residencial m²:', round(res['superficie_poligono_m2'].sum(), 2))

res.head()

In [ ]:
# Agrupamos por municipio
suelo_resumen = (
    res
    .groupby('codigo_ine')
    .agg(
        n_poligonos_residenciales=('OBJECTID', 'count'),
        superficie_residencial_m2=('superficie_poligono_m2', 'sum'),
        superficie_media_poligono_m2=('superficie_poligono_m2', 'mean'),
        superficie_mediana_poligono_m2=('superficie_poligono_m2', 'median'),
        superficie_max_poligono_m2=('superficie_poligono_m2', 'max'),
        perimetro_total_m=('perimetro_m', 'sum')
    )
    .reset_index()
)

suelo_resumen['superficie_residencial_ha'] = suelo_resumen['superficie_residencial_m2'] / 10_000

suelo_resumen = suelo_resumen.sort_values('superficie_residencial_m2', ascending=False)
suelo_resumen.head(15)

In [ ]:
# ============================================================
# 4. CRUCE SUELO RESIDENCIAL + MUNICIPIOS INE
# ============================================================

dataset = suelo_resumen.merge(
    municipios,
    on='codigo_ine',
    how='left',
    validate='many_to_one'
)

# Reordenamos columnas
cols = [
    'codigo_ine', 'municipio', 'poblacion_2025',
    'n_poligonos_residenciales',
    'superficie_residencial_m2', 'superficie_residencial_ha',
    'superficie_media_poligono_m2', 'superficie_mediana_poligono_m2',
    'superficie_max_poligono_m2', 'perimetro_total_m'
]
dataset = dataset[cols]

# Variables derivadas útiles para EDA
# m² de suelo residencial por habitante
# cuidado: si población es 0 o nula, evitamos división inválida
dataset['suelo_residencial_m2_por_hab'] = np.where(
    dataset['poblacion_2025'] > 0,
    dataset['superficie_residencial_m2'] / dataset['poblacion_2025'],
    np.nan
)

# Orden final
dataset = dataset.sort_values('superficie_residencial_m2', ascending=False).reset_index(drop=True)

dataset.head(20)

In [ ]:
# ============================================================
# 5. CONTROL DE CALIDAD DEL CRUCE
# ============================================================

sin_nombre = dataset[dataset['municipio'].isna()]
print('Municipios sin nombre tras cruce:', len(sin_nombre))

if len(sin_nombre) > 0:
    display(sin_nombre[['codigo_ine', 'superficie_residencial_m2', 'n_poligonos_residenciales']])
else:
    print('Cruce correcto: todos los códigos INE del suelo residencial tienen municipio asociado.')

In [ ]:
# Municipios clave para la demo
municipios_demo = [
    'Granada', 'Motril', 'Salobreña', 'Almuñécar',
    'Armilla', 'Maracena', 'Albolote', 'Atarfe',
    'Las Gabias', 'Ogíjares', 'Loja', 'Guadix', 'Baza'
]

# Búsqueda flexible por si hay tildes o nombres compuestos diferentes
patron = '|'.join(municipios_demo)

demo = dataset[dataset['municipio'].fillna('').str.contains(patron, case=False, regex=True)].copy()
demo.sort_values('superficie_residencial_m2', ascending=False)

In [ ]:
# ============================================================
# 6. EXPORTACIÓN DE RESULTADOS
# ============================================================

municipios.to_csv(DATA_PROCESSED / 'municipios_granada.csv', index=False, encoding='utf-8-sig')
suelo_resumen.to_csv(DATA_PROCESSED / 'suelo_residencial_municipio.csv', index=False, encoding='utf-8-sig')
dataset.to_csv(DATA_PROCESSED / 'dataset_maestro_granada_v1.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('-', DATA_PROCESSED / 'municipios_granada.csv')
print('-', DATA_PROCESSED / 'suelo_residencial_municipio.csv')
print('-', DATA_PROCESSED / 'dataset_maestro_granada_v1.csv')

In [ ]:
# ============================================================
# 7. RESUMEN EJECUTIVO PARA DOCUMENTACIÓN
# ============================================================

resumen = {
    'municipios_ine_total': municipios.shape[0],
    'poligonos_residenciales_total': res.shape[0],
    'municipios_con_suelo_residencial': dataset.shape[0],
    'superficie_residencial_total_m2': round(dataset['superficie_residencial_m2'].sum(), 2),
    'superficie_residencial_total_ha': round(dataset['superficie_residencial_ha'].sum(), 2),
    'poblacion_total_municipios_con_suelo': int(dataset['poblacion_2025'].sum())
}

resumen

## Siguiente paso

Una vez generado `dataset_maestro_granada_v1.csv`, el siguiente notebook será:

`02_eda_granada.ipynb`

Ahí analizaremos:

- Distribución de superficie residencial por municipio.
- Ranking de municipios.
- Relación entre población y suelo residencial.
- Municipios con mayor suelo residencial por habitante.
- Selección de municipios prioritarios para la demo.
